<a href="https://colab.research.google.com/github/soil7/soil7/blob/main/ICS_3Latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

# Loading train dataset
train = pd.read_csv('/content/train.csv')
num_samples_train = train.shape[0]

# To Check 'label' column exists
if 'label' in train.columns:
    num_labelled = train['label'].notnull().sum()  # Count labeled samples
    num_unlabelled = train['label'].isnull().sum()  # Count missing labels

    # Check label distribution
    label_distribution = train['label'].value_counts(normalize=True).to_dict()
else:
    num_labelled = 0
    num_unlabelled = num_samples_train
    label_distribution = "Label column missing!"

#  Dataset summary
dataset_summary = {
    "Total Training Samples": num_samples_train,
    "Labelled Data": num_labelled,
    "Unlabelled Data": num_unlabelled,
    "Label Distribution": label_distribution
}

#  Dataset summary as dataframe
df_summary = pd.DataFrame.from_dict(dataset_summary, orient='index', columns=['Value'])
print("\nDataset Summary:")
print(df_summary)



Dataset Summary:
                                             Value
Total Training Samples                       50000
Labelled Data                                50000
Unlabelled Data                                  0
Label Distribution      {1.0: 0.5313, 0.0: 0.4687}


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  StandardScaler
import pandas as pd
import numpy as np

# Manually Upload test.csv
uploaded = files.upload()

#  Loading the dataset
df = pd.read_csv('/content/train.csv')

# Handling Missing Data (Before Splitting)
df.fillna(df.median(), inplace=True)  # Apply median imputation

# Splitting into Train & Validation Sets (80% Train, 20% Validation)
X = df.drop(columns=['label'])  # Features
y = df['label']  # Target variable

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Applying Feature Scaling (Fit on Training Set)
scaler = StandardScaler()  # Use StandardScaler

X_train_scaled = scaler.fit_transform(X_train)  # Fit & transform on training data
X_val_scaled = scaler.transform(X_val)  # Transform validation data (NO fitting!)]


# Convert the Scaled Data  to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X.columns)

print(f" Training Set: {X_train_scaled.shape[0]} samples")
print(f"Validation Set: {X_val_scaled.shape[0]} samples")

 Training Set: 40000 samples
Validation Set: 10000 samples


In [9]:
print("Columns in train dataset:", list(train.columns))

Columns in train dataset: ['label', 'f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27']


In [10]:
print(train.head())  # Preview first few rows
print(train.info())  # Check data structure

   label        f0        f1        f2        f3        f4        f5  \
0    1.0  0.869293 -0.635082  0.225690  0.327470 -0.689993  0.754202   
1    1.0  0.907542  0.329147  0.359412  1.497970 -0.313010  1.095531   
2    1.0  0.798835  1.470639 -1.635975  0.453773  0.425629  1.104875   
3    0.0  1.344385 -0.876626  0.935913  1.992050  0.882454  1.786066   
4    1.0  1.105009  0.321356  1.522401  0.882808 -1.205349  0.681466   

         f6        f7        f8  ...       f18       f19       f20       f21  \
0 -0.248573 -1.092064  0.000000  ... -0.010455 -0.045767  3.101961  1.353760   
1 -0.557525 -1.588230  2.173076  ... -1.138930 -0.000819  0.000000  0.302220   
2  1.282322  1.381664  0.000000  ...  1.128848  0.900461  0.000000  0.909753   
3 -1.646778 -0.942383  0.000000  ... -0.678379 -1.360356  0.000000  0.946652   
4 -1.070464 -0.921871  0.000000  ... -0.373566  0.113041  0.000000  0.755856   

        f22       f23       f24       f25       f26       f27  
0  0.979563  0.978076 

In [11]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from google.colab import files

# 📂 Upload test.csv manually in Google Colab
uploaded = files.upload()

# Load Test Data
test = pd.read_csv('/content/test.csv')

# Handle Missing Data (Impute Using Median)
test.fillna(test.median(), inplace=True)

# Apply Standardization Using the SAME Scaler Used on Training Data
X_test_scaled = scaler.transform(test)  # Use the scaler already fitted on X_train

# Convert Processed Data into DataFrame
X_test_scaled = pd.DataFrame(X_test_scaled, columns=test.columns)

# Save Preprocessed Test Data
X_test_scaled.to_csv('/content/test_scaled.csv', index=False)



In [12]:
# Apply Standardization (Using Training Data Statistics)
print("Shape of X_test:", X_test_scaled.shape)


Shape of X_test: (50000, 28)


**XGBClassifier**

In [13]:
#  Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed
# Step 2: Import Libraries
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

#  Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)),  # Balanced RF
    ('xgb', XGBClassifier(n_estimators=100, max_depth=8, learning_rate=0.07, random_state=42)),  # Tuned XGBoost
    ('lgbm', LGBMClassifier(n_estimators=100, learning_rate=0.06, random_state=42)),  # Tuned LightGBM
    ('cat', CatBoostClassifier(iterations=100, learning_rate=0.06, verbose=0, random_state=42))  # Tuned CatBoost
]

#  Stacking Model with XGBoost as Final Estimator
ensemble_model = StackingClassifier(
    estimators=base_models,
    final_estimator=XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42),
    passthrough=True  # Allow base model predictions as features
)

#  Train the Model
ensemble_model.fit(X_train_scaled, y_train)

#  Make Predictions
y_pred = ensemble_model.predict(X_val_scaled)
y_pred_prob = ensemble_model.predict_proba(X_val_scaled)[:, 1]

#  Evaluate Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f"Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f"Stacking Ensemble AUROC: {auroc:.4f}")

[LightGBM] [Info] Number of positive: 21252, number of negative: 18748
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002496 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531300 -> initscore=0.125364
[LightGBM] [Info] Start training from score 0.125364
[LightGBM] [Info] Number of positive: 17002, number of negative: 14998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002821 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531312 -> initscore=0.125414
[Ligh

In [14]:
#  Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed

#  Import Libraries
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

#  Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42)),  # Balanced RF
    ('xgb', XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.07, random_state=42)),  # Tuned XGBoost
    ('lgbm', LGBMClassifier(n_estimators=300, learning_rate=0.06, random_state=42)),  # Tuned LightGBM
    ('cat', CatBoostClassifier(iterations=300, learning_rate=0.06, verbose=0, random_state=42))  # Tuned CatBoost
]

#  Defining Stacking Model with XGBoost as Final Estimator
ensemble_model = StackingClassifier(
    estimators=base_models,
    final_estimator=XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42),
    passthrough=True  # Allow base model predictions as features
)

#  Train the Model
ensemble_model.fit(X_train_scaled, y_train)

#  Make Predictions
y_pred = ensemble_model.predict(X_val_scaled)
y_pred_prob = ensemble_model.predict_proba(X_val_scaled)[:, 1]

#  Evaluate Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f" Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f" Stacking Ensemble AUROC: {auroc:.4f}")

[LightGBM] [Info] Number of positive: 21252, number of negative: 18748
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002589 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531300 -> initscore=0.125364
[LightGBM] [Info] Start training from score 0.125364
[LightGBM] [Info] Number of positive: 17002, number of negative: 14998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002225 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531312 -> initscore=0.125414
[LightGBM] [Info] Start training from score 0.125414
[LightGBM] [Info

In [15]:
#  Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed

#  Import Libraries
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

#  Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=600, max_depth=12, random_state=42)),  # Balanced RF
    ('xgb', XGBClassifier(n_estimators=600, max_depth=8, learning_rate=0.07, random_state=42)),  # Tuned XGBoost
    ('lgbm', LGBMClassifier(n_estimators=600, learning_rate=0.06, random_state=42)),  # Tuned LightGBM
    ('cat', CatBoostClassifier(iterations=600, learning_rate=0.06, verbose=0, random_state=42))  # Tuned CatBoost
]

#  Defining Stacking Model with XGBoost as Final Estimator
ensemble_model = StackingClassifier(
    estimators=base_models,
    final_estimator=XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42),
    passthrough=True  # Allow base model predictions as features
)

#  Train the Model
ensemble_model.fit(X_train_scaled, y_train)

#  Make Predictions
y_pred = ensemble_model.predict(X_val_scaled)
y_pred_prob = ensemble_model.predict_proba(X_val_scaled)[:, 1]

#  Evaluate Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f" Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f" Stacking Ensemble AUROC: {auroc:.4f}")

[LightGBM] [Info] Number of positive: 21252, number of negative: 18748
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003111 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531300 -> initscore=0.125364
[LightGBM] [Info] Start training from score 0.125364
[LightGBM] [Info] Number of positive: 17002, number of negative: 14998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002245 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531312 -> initscore=0.125414
[LightGBM] [Info] Start training from score 0.125414
[LightGBM] [Info

**GradientBoostingClassifier**

In [16]:
#  Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed

#  Import Libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

#  Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)),  # Random Forest
    ('xgb', XGBClassifier(n_estimators=100, max_depth=15, learning_rate=0.08, random_state=42)),  # XGBoost
    ('lgbm', LGBMClassifier(n_estimators=100, learning_rate=0.07, random_state=42)),  # LightGBM
    ('cat', CatBoostClassifier(iterations=100, learning_rate=0.07, verbose=0, random_state=42))  # CatBoost
]

# Define Stacking Model with GradientBoosting Final Estimator
ensemble_model2 = StackingClassifier(
    estimators=base_models,
    final_estimator=GradientBoostingClassifier(n_estimators=50, learning_rate=0.05, random_state=42),
    passthrough=True  # Allow base model predictions as features
)
# Cross-Validation Strategy (StratifiedKFold)
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)  # 2-Fold Cross-Validation

# Perform Cross-Validation
cv_scores = cross_val_score(
    ensemble_model2, X_train_scaled, y_train,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f" Cross-Validation AUROC Scores: {cv_scores}")
print(f" Mean AUROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


#  Train the Model
ensemble_model2.fit(X_train_scaled, y_train)

#  Make Predictions on validation set
y_pred = ensemble_model2.predict(X_val_scaled)
y_pred_prob = ensemble_model2.predict_proba(X_val_scaled)[:, 1]

#  Evaluate Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f" Stronger Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f" Stronger Stacking Ensemble AUROC: {auroc:.4f}")

 Cross-Validation AUROC Scores: [0.7937407  0.79391026]
 Mean AUROC: 0.7938 ± 0.0001
[LightGBM] [Info] Number of positive: 21252, number of negative: 18748
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002783 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531300 -> initscore=0.125364
[LightGBM] [Info] Start training from score 0.125364
[LightGBM] [Info] Number of positive: 17002, number of negative: 14998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002431 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531312 -> ini

In [17]:
#  Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed

#  Import Libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

#  Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=18, random_state=42)),  # Random Forest
    ('xgb', XGBClassifier(n_estimators=300, max_depth=18, learning_rate=0.08, random_state=42)),  # XGBoost
    ('lgbm', LGBMClassifier(n_estimators=300, learning_rate=0.06, random_state=42)),  # LightGBM
    ('cat', CatBoostClassifier(iterations=300, learning_rate=0.06, verbose=0, random_state=42))  # CatBoost
]

#  Define Stacking Model with GradientBoosting Final Estimator
ensemble_model2 = StackingClassifier(
    estimators=base_models,
    final_estimator=GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    passthrough=True  # Allow base model predictions as features
)

#  Cross-Validation Strategy (StratifiedKFold)
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)  # 2-Fold Cross-Validation

#   Perform Cross-Validation
cv_scores = cross_val_score(
    ensemble_model2, X_train_scaled, y_train,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f" Cross-Validation AUROC Scores: {cv_scores}")
print(f" Mean AUROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Train the Model
ensemble_model2.fit(X_train_scaled, y_train)

#  Make Predictions on validation set
y_pred = ensemble_model2.predict(X_val_scaled)
y_pred_prob = ensemble_model2.predict_proba(X_val_scaled)[:, 1]

#  Evaluate Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f" Stronger Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f" Stronger Stacking Ensemble AUROC: {auroc:.4f}")

 Cross-Validation AUROC Scores: [0.79764061 0.79763216]
 Mean AUROC: 0.7976 ± 0.0000
[LightGBM] [Info] Number of positive: 21252, number of negative: 18748
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002633 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531300 -> initscore=0.125364
[LightGBM] [Info] Start training from score 0.125364
[LightGBM] [Info] Number of positive: 17002, number of negative: 14998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002357 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531312 -> ini

In [18]:
#  Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed

#  Import Libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

#  Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=1200, max_depth=18, random_state=42)),  # Random Forest
    ('xgb', XGBClassifier(n_estimators=1200, max_depth=18, learning_rate=0.08, random_state=42)),  # XGBoost
    ('lgbm', LGBMClassifier(n_estimators=1200, learning_rate=0.06, random_state=42)),  # LightGBM
    ('cat', CatBoostClassifier(iterations=1200, learning_rate=0.06, verbose=0, random_state=42))  # CatBoost
]

#  Define Stacking Model with GradientBoosting Final Estimator
ensemble_model2 = StackingClassifier(
    estimators=base_models,
    final_estimator=GradientBoostingClassifier(n_estimators=45, learning_rate=0.05, random_state=42),
    passthrough=True  # Allow base model predictions as features
)

#  Cross-Validation Strategy (StratifiedKFold)
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)  # 5-Fold Cross-Validation

#   Perform Cross-Validation
cv_scores = cross_val_score(
    ensemble_model2, X_train_scaled, y_train,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f" Cross-Validation AUROC Scores: {cv_scores}")
print(f" Mean AUROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Train the Model
ensemble_model2.fit(X_train_scaled, y_train)

#  Make Predictions on validation set
y_pred = ensemble_model2.predict(X_val_scaled)
y_pred_prob = ensemble_model2.predict_proba(X_val_scaled)[:, 1]

#  Evaluate Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f" Stronger Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f" Stronger Stacking Ensemble AUROC: {auroc:.4f}")

 Cross-Validation AUROC Scores: [0.79742459 0.79758864]
 Mean AUROC: 0.7975 ± 0.0001
[LightGBM] [Info] Number of positive: 21252, number of negative: 18748
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000935 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531300 -> initscore=0.125364
[LightGBM] [Info] Start training from score 0.125364
[LightGBM] [Info] Number of positive: 17002, number of negative: 14998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003803 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6136
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 28

***Creating a submission file with using the nest model ***

In [19]:
import pandas as pd
import numpy as np

# Predict probabilities on test data
y_test_proba = ensemble_model2.predict_proba(X_test_scaled)  # Get probabilities for both classes

# Extract probabilities for the positive class (Class 1)
y_test_pred = y_test_proba[:, 1]  # Only take the probability of class 1

# Generate ID column
ids = np.arange(len(y_test_pred))  # Generate IDs starting from 0

# Create Submission DataFrame
submission_df = pd.DataFrame({'Id': ids, 'Predicted': y_test_pred})

# Ensure 'Id' is an integer (as Kaggle expects)
submission_df['Id'] = submission_df['Id'].astype(np.int64)

# Convert 'Id' to scientific notation (with 18 decimal places)
submission_df['Id'] = submission_df['Id'].apply(lambda x: f"{float(x):.18e}")

# Save to CSV (Ensure Proper Formatting)
submission_df.to_csv("submission ensemble2.csv", index=False)

# Show first few rows for verification
print(" Submission file saved as `submission.csv` (Ready for Kaggle Upload!)")
submission_df.head()

 Submission file saved as `submission.csv` (Ready for Kaggle Upload!)


,Id,Predicted
0,0.000000000000000000e+00,0.132015
1,1.000000000000000000e+00,0.491699
2,2.000000000000000000e+00,0.670277
3,3.000000000000000000e+00,0.830187
4,4.000000000000000000e+00,0.421258


In [20]:
# Install Missing Libraries
!pip install catboost lightgbm xgboost  # Ensure all necessary libraries are installed

# Import Libraries
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

# Cross-Validation Strategy (StratifiedKFold)
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)  # 2-Fold Cross-Validation

# Train & Evaluate Decision Tree with Different Depths
for depth in [50, 100, 150]:
    print(f"\n==== Training DecisionTreeClassifier with max_depth={depth} ====")

    # Define Decision Tree Model
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    # Perform Cross-Validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    print(f"Cross-Validation AUROC Scores: {cv_scores}")
    print(f"Mean AUROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # Train Decision Tree on Full Training Set
    model.fit(X_train_scaled, y_train)

    # Make Predictions on Validation Set
    y_pred = model.predict(X_val_scaled)
    y_pred_prob = model.predict_proba(X_val_scaled)[:, 1]

    # Evaluate Model Performance
    accuracy = accuracy_score(y_val, y_pred)
    auroc = roc_auc_score(y_val, y_pred_prob)

    print(f"Decision Tree Accuracy (max_depth={depth}): {accuracy:.4f}")
    print(f"Decision Tree AUROC (max_depth={depth}): {auroc:.4f}")


==== Training DecisionTreeClassifier with max_depth=50 ====
Cross-Validation AUROC Scores: [0.61497255 0.61600468]
Mean AUROC: 0.6155 ± 0.0005
Decision Tree Accuracy (max_depth=50): 0.6150
Decision Tree AUROC (max_depth=50): 0.6134

==== Training DecisionTreeClassifier with max_depth=100 ====
Cross-Validation AUROC Scores: [0.61497255 0.61600468]
Mean AUROC: 0.6155 ± 0.0005
Decision Tree Accuracy (max_depth=100): 0.6150
Decision Tree AUROC (max_depth=100): 0.6134

==== Training DecisionTreeClassifier with max_depth=150 ====
Cross-Validation AUROC Scores: [0.61497255 0.61600468]
Mean AUROC: 0.6155 ± 0.0005
Decision Tree Accuracy (max_depth=150): 0.6150
Decision Tree AUROC (max_depth=150): 0.6134


In [10]:
# Step 1: Install Missing Libraries
!pip install catboost lightgbm xgboost autogluon.tabular

# Step 2: Import Libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
from autogluon.tabular import TabularPredictor

# Step 3: Define Base Models (Balanced for Performance & Speed)
base_models = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=18, random_state=42)),  # Random Forest
    ('xgb', XGBClassifier(n_estimators=200, max_depth=18, learning_rate=0.08, random_state=42)),  # XGBoost
    ('lgbm', LGBMClassifier(n_estimators=200, learning_rate=0.06, random_state=42)),  # LightGBM
    ('cat', CatBoostClassifier(iterations=200, learning_rate=0.06, verbose=0, random_state=42))  # CatBoost
]

# Step 4: Define Stacking Model with GradientBoosting as Final Estimator
ensemble_model = StackingClassifier(
    estimators=base_models,
    final_estimator=XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42),
    passthrough=True
)



# Step 7: Train the Final Model on the Full Training Set
ensemble_model.fit(X_train_scaled, y_train)

# Step 8: Make Predictions on Validation Set
y_pred = ensemble_model.predict(X_val_scaled)
y_pred_prob = ensemble_model.predict_proba(X_val_scaled)[:, 1]

# Step 9: Evaluate Final Model Performance
accuracy = accuracy_score(y_val, y_pred)
auroc = roc_auc_score(y_val, y_pred_prob)

print(f"Stacking Ensemble Accuracy: {accuracy:.4f}")
print(f"Stacking Ensemble AUROC: {auroc:.4f}")

# Step 10: AutoGluon for Automated Model Selection & Training
train_data = pd.DataFrame(X_train_scaled, columns=[f'feature_{i}' for i in range(X_train_scaled.shape[1])])
train_data['label'] = y_train

test_data = pd.DataFrame(X_val_scaled, columns=[f'feature_{i}' for i in range(X_val_scaled.shape[1])])
test_data['label'] = y_val

# Fix Missing Values in Labels
train_data.dropna(subset=['label'], inplace=True)
test_data.dropna(subset=['label'], inplace=True)

# Train AutoGluon
predictor = TabularPredictor(label='label').fit(train_data, presets="best_quality")

# Step 11: Make AutoGluon Predictions
autogluon_preds = predictor.predict(test_data.drop(columns=['label']))
autogluon_prob = predictor.predict_proba(test_data.drop(columns=['label']))[1]

# Step 12: Evaluate AutoGluon Performance
autogluon_accuracy = accuracy_score(test_data['label'], autogluon_preds)
autogluon_auroc = roc_auc_score(test_data['label'], autogluon_prob)

print(f"AutoGluon Accuracy: {autogluon_accuracy:.4f}")
print(f"AutoGluon AUROC: {autogluon_auroc:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.2/352.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.2/266.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


ImportError: cannot import name '_add_to_diagonal' from 'sklearn.utils._array_api' (/usr/local/lib/python3.11/dist-packages/sklearn/utils/_array_api.py)